###  Main problem statement: “To determine which client segments generate the highest gross profit while maintaining strong customer satisfaction” (enabling the company to prioritize high-value clients, improve client retention, and support sustainable business growth)​

#### Key points (subproblems)​

#### Profitability and its drivers by client segment: Analyse gross profit and gross margin across client type, industry sector, organisation size and location, while examining hardware, software and manpower costs and service ratings to identify high-value segments and opportunities for cost optimisation and margin improvement.​

In [7]:
import pandas as pd
import plotly.express as px

# ============================================
# Load and Prepare Data
# ============================================
xls = pd.ExcelFile("merged.xlsx")
df_merged = pd.read_excel(xls, xls.sheet_names[0])

# Financial calculations
df_merged["COGS"] = (
    df_merged["HARDWARE"]
    + df_merged["SOFTWARE"]
    + df_merged["MANPOWER"]
)

df_merged["GROSS_PROFIT"] = (
    df_merged["REVENUE"]
    - df_merged["COGS"]
)

df_merged["GROSS_MARGIN"] = (
    df_merged["GROSS_PROFIT"]
    / df_merged["REVENUE"]
) * 100

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

frames = []

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)
        .agg({
            "GROSS_PROFIT": "sum",
            "GROSS_MARGIN": "mean",
            "REVENUE": "sum",
            "NPS RATING": "mean"
        })
        .reset_index()
    )

    grouped = grouped.rename(columns={col: "Segment"})
    grouped["Segmentation"] = label

    # Sort from highest to lowest profit
    grouped = grouped.sort_values(
        "GROSS_PROFIT",
        ascending=False
    )

    frames.append(grouped)

plot_df = pd.concat(frames, ignore_index=True)

# ============================================
# Animated Horizontal Bar Chart
# ============================================

fig = px.bar(

    plot_df,

    x="GROSS_PROFIT",

    y="Segment",

    orientation="h",

    color="GROSS_MARGIN",

    color_continuous_scale="Viridis",

    animation_frame="Segmentation",

    hover_name="Segment",

    hover_data={
        "GROSS_PROFIT": ":,.0f",
        "GROSS_MARGIN": ":.2f",
        "REVENUE": ":,.0f",
        "NPS RATING": ":.2f"
    },

    title="Gross Profit Across Client Segmentations"

)

fig.update_layout(

    template="plotly_white",

    title_x=0.5,

    xaxis_title="Total Gross Profit",

    yaxis_title="Client Segment",

    height=650,

    coloraxis_colorbar=dict(
        title="Gross Margin (%)"
    )

)

fig.show()
fig1 = fig

In [8]:
import plotly.graph_objects as go
import numpy as np

# ============================================
# Create Average Service Rating
# ============================================
service_cols = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT"
]

df_merged["AVG_SERVICE"] = df_merged[service_cols].mean(axis=1)

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

aggregated = {}

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)
        .agg({
            "AVG_SERVICE": "mean",
            "GROSS_MARGIN": "mean",
            "GROSS_PROFIT": "sum",
            "REVENUE": "sum"
        })
        .reset_index()
    )

    aggregated[label] = grouped

# ============================================
# Default View
# ============================================
default = "Industry Sector"

data = aggregated[default]
segment_col = segmentations[default]

# ============================================
# Bubble Size Scaling
# ============================================
bubble_size = (
    data["REVENUE"] / data["REVENUE"].max()
) * 60 + 12

# ============================================
# Figure
# ============================================
fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=data["AVG_SERVICE"],
        y=data["GROSS_MARGIN"],

        mode="markers+text",

        text=data[segment_col],
        textposition="top center",

        marker=dict(

            size=bubble_size,

            color=data["GROSS_PROFIT"],

            colorscale="Viridis",

            showscale=True,

            colorbar=dict(title="Gross Profit"),

            sizemode="diameter",

            line=dict(width=1)

        ),

        customdata=np.stack(

            (

                data[segment_col],

                data["GROSS_PROFIT"],

                data["REVENUE"]

            ),

            axis=-1

        ),

        hovertemplate=
        "<b>%{customdata[0]}</b><br><br>" +
        "Average Service Rating: %{x:.2f}<br>" +
        "Gross Margin: %{y:.2f}%<br>" +
        "Gross Profit: %{customdata[1]:,.0f}<br>" +
        "Revenue: %{customdata[2]:,.0f}<extra></extra>"

    )

)

# ============================================
# Dropdown
# ============================================
buttons = []

for label, column in segmentations.items():

    temp = aggregated[label]

    bubble_size = (
        temp["REVENUE"] / temp["REVENUE"].max()
    ) * 60 + 12

    buttons.append(

        dict(

            label=label,

            method="update",

            args=[

                {

                    "x":[temp["AVG_SERVICE"]],

                    "y":[temp["GROSS_MARGIN"]],

                    "text":[temp[column]],

                    "customdata":[

                        np.stack(

                            (

                                temp[column],

                                temp["GROSS_PROFIT"],

                                temp["REVENUE"]

                            ),

                            axis=-1

                        )

                    ],

                    "marker":[

                        dict(

                            size=bubble_size,

                            color=temp["GROSS_PROFIT"],

                            colorscale="Viridis",

                            showscale=True,

                            colorbar=dict(title="Gross Profit"),

                            sizemode="diameter",

                            line=dict(width=1)

                        )

                    ]

                },

                {

                    "title":f"Service Quality vs Gross Margin by {label}",

                    "xaxis":{"title":"Average Service Rating"},

                    "yaxis":{"title":"Average Gross Margin (%)"}

                }

            ]

        )

    )

# ============================================
# Layout
# ============================================
fig.update_layout(

    title="Service Quality vs Gross Margin by Industry Sector",

    template="plotly_white",

    xaxis_title="Average Service Rating",

    yaxis_title="Average Gross Margin (%)",

    updatemenus=[

        dict(

            buttons=buttons,

            direction="down",

            x=0.02,

            y=1.18,

            showactive=True

        )

    ]

)

fig.show()
fig2 = fig

In [9]:
import plotly.graph_objects as go
import pandas as pd
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

aggregated = {}

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)[["HARDWARE", "SOFTWARE", "MANPOWER"]]
        .sum()
        .reset_index()
    )

    aggregated[label] = grouped
default = "Industry Sector"

data = aggregated[default]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["HARDWARE"],
        name="Hardware"
    )
)

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["SOFTWARE"],
        name="Software"
    )
)

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["MANPOWER"],
        name="Manpower"
    )
)
buttons = []

for label, column in segmentations.items():

    temp = aggregated[label]

    buttons.append(

        dict(

            label=label,

            method="update",

            args=[

                {
                    "x":[
                        temp[column],
                        temp[column],
                        temp[column]
                    ],

                    "y":[
                        temp["HARDWARE"],
                        temp["SOFTWARE"],
                        temp["MANPOWER"]
                    ]

                },

                {
                    "title":f"Cost Composition by {label}",
                    "xaxis":{"title":label}
                }

            ]

        )

    )
mode_buttons = [

    dict(

        label="Stacked",

        method="relayout",

        args=[{"barmode":"stack"}]

    ),

    dict(

        label="Grouped",

        method="relayout",

        args=[{"barmode":"group"}]

    )

]
fig.update_layout(

    title="Cost Composition by Industry Sector",

    xaxis_title="Industry Sector",

    yaxis_title="Total Cost",

    barmode="stack",

    template="plotly_white",

    updatemenus=[

        dict(

            buttons=buttons,

            direction="down",

            x=0.02,

            y=1.18,

            showactive=True

        ),

        dict(

            buttons=mode_buttons,

            direction="right",

            x=0.55,

            y=1.18,

            showactive=True

        )

    ]

)

fig.show()
fig3 = fig

In [8]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ---------------------------
# Dashboard data and helpers
# ---------------------------
np.random.seed(42)
clients = [f"Client {chr(65+i)}" for i in range(16)]
data = []

for client_id in clients:
    revenue = np.random.uniform(50000, 250000)
    labor_cost = revenue * np.random.uniform(0.3, 0.45)
    overhead_cost = revenue * np.random.uniform(0.1, 0.2)
    material_cost = revenue * np.random.uniform(0.05, 0.15)
    gross_profit = revenue - (labor_cost + overhead_cost + material_cost)
    satisfaction_score = np.random.uniform(5.5, 9.8)
    industry = np.random.choice(["Tech", "Finance", "Healthcare", "Retail"])
    client_type = np.random.choice(["Enterprise", "SMB", "Government"])
    
    data.append({
        "Client": client_id,
        "Industry": industry,
        "Client_Type": client_type,
        "Revenue": round(revenue, 2),
        "Labor_Cost": round(labor_cost, 2),
        "Overhead_Cost": round(overhead_cost, 2),
        "Material_Cost": round(material_cost, 2),
        "Gross_Profit": round(gross_profit, 2),
        "Gross_Margin": gross_profit / revenue,
        "Service_Quality_Score": round(satisfaction_score, 1)
    })

df = pd.DataFrame(data)

# Control Options
industry_options = ['All'] + sorted(df['Industry'].unique().tolist())
metric_options = [
    {'label': 'Gross Profit', 'value': 'Gross_Profit'},
    {'label': 'Revenue', 'value': 'Revenue'}
]
metric_labels = {'Gross_Profit': 'Gross Profit', 'Revenue': 'Revenue'}


def _filtered_clients(industry_value, min_quality):
    """Filter clients based on dropdown and slider selections."""
    d = df.copy()
    if industry_value != 'All':
        d = d[d['Industry'] == industry_value]
    d = d[d['Service_Quality_Score'] >= min_quality]
    return d


def dashboard_fig1(d, metric_value):
    """Chart 1: Plotly Express Ranked Profitability / Revenue Chart."""
    if d.empty:
        return go.Figure().update_layout(title='No clients match the selected filters', template='plotly_white')
    
    d_sorted = d.sort_values(metric_value, ascending=True)
    
    f = px.bar(
        d_sorted,
        x=metric_value,
        y='Client',
        color='Gross_Margin',
        color_continuous_scale='Viridis',
        orientation='h',
        template='plotly_white',
        hover_data=['Industry', 'Client_Type', 'Revenue', 'Gross_Profit', 'Service_Quality_Score']
    )
    
    f.update_layout(
        title=f'{metric_labels[metric_value]} Overview by Client',
        xaxis_title=metric_labels[metric_value],
        yaxis_title='',
        height=450,
        margin=dict(t=60, l=100, r=40, b=50),
        coloraxis_colorbar=dict(title="Margin (%)", tickformat=".0%")
    )
    f.update_xaxes(tickformat='$,.0f')
    return f


def dashboard_fig2(d):
    """Chart 2: Graph Objects Service Quality vs Profitability Scatter Matrix."""
    if d.empty:
        return go.Figure().update_layout(title='No clients match the selected filters', template='plotly_white')
    
    f = go.Figure()
    
    for ind in d['Industry'].unique():
        x = d[d['Industry'] == ind]
        custom = x[['Client', 'Client_Type', 'Revenue', 'Gross_Profit', 'Gross_Margin']].to_numpy()
        
        f.add_trace(go.Scatter(
            x=x['Service_Quality_Score'],
            y=x['Gross_Margin'] * 100,
            mode='markers',
            name=ind,
            marker=dict(
                size=np.clip(np.sqrt(x['Revenue'].clip(lower=0)) / 15, 10, 40),
                opacity=0.8,
                line=dict(width=1, color='white')
            ),
            customdata=custom,
            hovertemplate='<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
                          'Quality Score: %{x:.1f}/10<br>'
                          'Gross Margin: %{y:.1f}%<br>'
                          'Revenue: $%{customdata[2]:,.0f}<br>'
                          'Gross Profit: $%{customdata[3]:,.0f}<extra></extra>'
        ))
        
    f.add_vline(x=d['Service_Quality_Score'].median(), line_dash='dash', line_color='#A0AEC0',
                annotation_text='Median Quality', annotation_position='top right')
    f.add_hline(y=d['Gross_Margin'].median() * 100, line_dash='dash', line_color='#A0AEC0',
                annotation_text='Median Margin', annotation_position='bottom right')

    f.update_layout(
        title='Service Quality vs Profitability Margin',
        template='plotly_white',
        height=450,
        xaxis=dict(title='Service Quality Score (1–10)', range=[5, 10.2]),
        yaxis=dict(title='Gross Margin (%)'),
        legend_title='Industry',
        margin=dict(t=60, l=60, r=30, b=50)
    )
    return f


def dashboard_fig3(d):
    """Chart 3: Graph Objects Stacked Cost Composition Chart."""
    if d.empty:
        return go.Figure().update_layout(title='No clients match the selected filters', template='plotly_white')
    
    f = go.Figure()
    
    f.add_trace(go.Bar(x=d['Client'], y=d['Labor_Cost'], name='Labor Cost', marker_color='#3182CE'))
    f.add_trace(go.Bar(x=d['Client'], y=d['Overhead_Cost'], name='Overhead Cost', marker_color='#DD6B20'))
    f.add_trace(go.Bar(x=d['Client'], y=d['Material_Cost'], name='Material Cost', marker_color='#38A169'))
    
    f.update_layout(
        title='Cost Composition Breakdown by Client',
        barmode='stack',
        template='plotly_white',
        height=450,
        xaxis_title='',
        yaxis_title='Cost ($)',
        legend_title='Cost Type',
        margin=dict(t=60, l=70, r=30, b=50)
    )
    f.update_yaxes(tickformat='$,.0f')
    return f


# ---------------------------
# Presentation-ready dashboard
# ---------------------------
app = Dash(__name__)
app.title = 'Client Profitability Dashboard'

logo_mark = html.Div(
    '$',
    style={
        'width': '42px', 'height': '42px', 'borderRadius': '10px',
        'backgroundColor': '#2B6CB0', 'color': 'white', 'fontSize': '24px',
        'fontWeight': '700', 'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'
    }
)

app.layout = html.Div([
    # Header Banner
    html.Div([
        html.Div([
            logo_mark,
            html.Div([
                html.Div('FINANCIAL ANALYTICS', style={'fontSize': '11px', 'fontWeight': '800', 'letterSpacing': '1.5px', 'color': '#2B6CB0'}),
                html.Div('Client Profitability & Performance', style={'fontSize': '20px', 'fontWeight': '700', 'color': '#1A202C'})
            ])
        ], style={'display': 'flex', 'alignItems': 'center', 'gap': '14px'}),
    ], style={'padding': '18px 28px', 'backgroundColor': 'white', 'borderBottom': '1px solid #E2E8F0'}),

    # Sub-header Title Block
    html.Div([
        html.H1('Client Performance & Cost Intelligence', style={'margin': '0', 'fontSize': '26px', 'color': '#1A202C'}),
        html.P('Analyze client margins, quality scores, and operational cost structures across industries.',
               style={'margin': '4px 0 0', 'color': '#4A5568', 'fontSize': '14px'})
    ], style={'padding': '16px 28px', 'backgroundColor': '#FFFFFF', 'borderBottom': '1px solid #E2E8F0'}),

    # Controls Section
    html.Div([
        html.Div([
            html.Label('Industry Sector', style={'fontWeight': '700', 'fontSize': '13px', 'color': '#2D3748'}),
            dcc.Dropdown(id='industry-control', options=[{'label': i, 'value': i} for i in industry_options], value='All', clearable=False)
        ]),
        html.Div([
            html.Label('Chart 1 Measure', style={'fontWeight': '700', 'fontSize': '13px', 'color': '#2D3748'}),
            dcc.RadioItems(
                id='metric-control',
                options=metric_options,
                value='Gross_Profit',
                inline=True,
                style={'marginTop': '8px'},
                labelStyle={'display': 'inline-block', 'marginRight': '16px', 'fontSize': '14px', 'color': '#2D3748'}
            )
        ]),
        html.Div([
            html.Label('Min Service Quality Score', style={'fontWeight': '700', 'fontSize': '13px', 'color': '#2D3748'}),
            dcc.Slider(
                id='quality-control', min=5.0, max=10.0, step=0.5, value=5.0,
                marks={i: str(i) for i in range(5, 11)},
                tooltip={'placement': 'bottom', 'always_visible': False}
            )
        ], style={'gridColumn': 'span 2'}),
    ], style={
        'display': 'grid', 'gridTemplateColumns': '1fr 1fr 2fr', 'gap': '16px 24px',
        'padding': '18px 28px', 'backgroundColor': '#F7FAFC', 'borderBottom': '1px solid #E2E8F0'
    }),

    # Dynamic KPI Summary Bar
    html.Div(id='kpi-container', style={
        'display': 'grid', 'gridTemplateColumns': 'repeat(4, 1fr)', 'gap': '16px',
        'padding': '20px 28px 4px 28px', 'backgroundColor': '#EDF2F7'
    }),

    # Charts Grid Layout
    html.Div([
        html.Div([dcc.Graph(id='dashboard-chart-1')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'}),
        html.Div([
            html.Div([dcc.Graph(id='dashboard-chart-2')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'}),
            html.Div([dcc.Graph(id='dashboard-chart-3')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'}),
        ], style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '18px'})
    ], style={'display': 'grid', 'gridTemplateColumns': '1fr', 'gap': '18px', 'padding': '16px 28px 24px 28px', 'backgroundColor': '#EDF2F7'}),

    # Footer
    html.Footer(
        'Financial analysis dashboard: Internal operations and account profitability metrics.',
        style={'padding': '14px 28px', 'fontSize': '12px', 'color': '#718096', 'backgroundColor': 'white', 'borderTop': '1px solid #E2E8F0'}
    )
], style={'fontFamily': 'Segoe UI, Arial, sans-serif', 'backgroundColor': '#EDF2F7', 'minHeight': '100vh'})


@app.callback(
    [
        Output('kpi-container', 'children'),
        Output('dashboard-chart-1', 'figure'),
        Output('dashboard-chart-2', 'figure'),
        Output('dashboard-chart-3', 'figure')
    ],
    [
        Input('industry-control', 'value'),
        Input('metric-control', 'value'),
        Input('quality-control', 'value')
    ]
)
def update_dashboard(industry_value, metric_value, min_quality):
    d = _filtered_clients(industry_value, min_quality)
    
    # Generate KPI summary cards
    if not d.empty:
        total_rev = d['Revenue'].sum()
        total_profit = d['Gross_Profit'].sum()
        avg_margin = d['Gross_Margin'].mean() * 100
        avg_quality = d['Service_Quality_Score'].mean()
    else:
        total_rev, total_profit, avg_margin, avg_quality = 0, 0, 0, 0

    kpi_metrics = [
        ("Total Revenue", f"${total_rev:,.0f}"),
        ("Total Gross Profit", f"${total_profit:,.0f}"),
        ("Avg Gross Margin", f"{avg_margin:.1f}%"),
        ("Avg Quality Score", f"{avg_quality:.1f} / 10")
    ]
    
    kpi_cards = [
        html.Div([
            html.Div(title, style={'fontSize': '12px', 'color': '#718096', 'fontWeight': '600'}),
            html.Div(val, style={'fontSize': '22px', 'fontWeight': '700', 'color': '#2B6CB0', 'marginTop': '4px'})
        ], style={'backgroundColor': 'white', 'padding': '16px 20px', 'borderRadius': '8px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'})
        for title, val in kpi_metrics
    ]
    
    return kpi_cards, dashboard_fig1(d, metric_value), dashboard_fig2(d), dashboard_fig3(d)


if __name__ == '__main__':
    app.run(debug=True)

### Insight
**Healthcare**, **Info Tech**, and **Manufacturing** are the strongest-performing sectors because they combine:
* High gross profit
* Strong gross margins

This means they are not only generating large amounts of revenue but are also converting revenue into profit efficiently.

---

### Action
The company should:
* Prioritise acquiring and retaining clients in these sectors.
* Allocate more sales and marketing resources to these industries.
* Develop sector-specific offerings for these clients.

### 2. Which sectors have margin improvement opportunities?
*(Using Graph 1 + Graph 3)*

* **Finance**
  * **Margin:** ~43% (good)
  * **Profit:** Relatively low (~4M)
  * *Takeaway:* Finance appears efficient but small in scale.

* **Education**
  * **Margin:** ~38%
  * **Profit:** ~10M
  * *Takeaway:* Moderate profitability with room for improvement.

* **Charity**
  * **Margin:** ~14%
  * **Profit:** Almost negligible
  * *Takeaway:* This sector contributes little profit and has poor margins.

---

### Action

**For low-margin sectors:**
* Review pricing strategy.
* Reduce unnecessary costs.
* Reassess whether these sectors should remain a strategic focus.

**For Charity specifically:**
* Consider whether the sector is strategically valuable.
* If retained, simplify service offerings to improve profitability.

### 3. What drives costs?
*(Using Graph 3)*

Several sectors show that:
* **Manpower** is consistently the largest cost component.
* **Software** is generally the second-largest.
* **Hardware** is often the smallest contributor.

#### Examples
* **Healthcare:** Very high manpower and software costs.
* **Manufacturing:** Large manpower and software expenditure.
* **Info Tech:** Heavy spending across all three categories.

---

### Insight
Profitability is strongly influenced by manpower efficiency. This suggests:
* Staff allocation
* Project productivity
* Resource utilisation

...are major drivers of margin performance.

---

### Action
Possible strategies:
* Improve workforce planning.
* Increase automation.
* Standardise project delivery processes.
* Reduce repetitive manual work.

> **Note:** Even a small reduction in manpower costs could significantly improve margins.

### 4. Does service quality affect profitability?
*(Using Graph 2)*

Graph 2 shows:

| Sector | Service | Margin |
| :--- | :--- | :--- |
| **Finance** | Highest service | Good margin |
| **Transportation** | High service | Highest margin |
| **Info Tech** | High service | Highest margin |
| **Manufacturing** | Good service | Highest margin |

Meanwhile:

| Sector | Service | Margin |
| :--- | :--- | :--- |
| **Charity** | High service | Low margin |

---

### Insight
There appears to be a generally positive relationship between service quality and profitability:
* Sectors with better service ratings often achieve stronger margins.

This suggests that investments in:
* Presales
* Technical expertise
* Delivery quality
* Post-sales support

...may contribute to better financial outcomes.

> **Note:** Charity shows that good service alone does not guarantee profitability, indicating that pricing and cost structures also matter.

### Summary & Conclusion

**Healthcare**, **Info Tech**, and **Manufacturing** are the most attractive client segments because they generate the highest gross profits while maintaining strong gross margins and above-average service ratings.

These sectors should therefore be prioritised for:
* Client acquisition
* Retention initiatives
* Additional investment

---

#### Key Takeaways
* **Cost Optimisation:** Efforts should focus primarily on manpower efficiency, particularly in sectors with high labour costs but weaker margins.
* **Service Quality:** The positive relationship observed between service quality and profitability suggests that maintaining strong customer experience can support both client retention and sustainable business growth.